In [1]:
# 1. Instalar herramientas para guardar imágenes
!pip install -q altair vl-convert-python

import pandas as pd
import altair as alt

# 2. Cargar tu archivo CSV limpio
df = pd.read_csv("Transfers_cleaned2.csv", sep=";")

# 3. Calcular el sobreprecio acumulado por club (en millones de euros)
df["sobreprecio_eur"] = df["Valor pagado"] - df["Valor estimado"]

resumen_clubes = (
    df.groupby("Club comprador_2")
    .agg(
        total_pagado=("Valor pagado", "sum"),
        total_estimado=("Valor estimado", "sum"),
        sobreprecio_total=("sobreprecio_eur", "sum"),
        sobreprecio_pct_promedio=("Deficit", "mean"),
        total_fichajes=("Jugador", "count"),
    )
    .reset_index()
)

# Convertir cifras a millones de euros (€M)
resumen_clubes["Sobreprecio (€M)"] = (
    resumen_clubes["sobreprecio_total"] / 1_000_000
).round(1)
resumen_clubes["Total Pagado (€M)"] = (
    resumen_clubes["total_pagado"] / 1_000_000
).round(1)
resumen_clubes["Valor Estimado Real (€M)"] = (
    resumen_clubes["total_estimado"] / 1_000_000
).round(1)
resumen_clubes["Sobreprecio Promedio (%)"] = resumen_clubes[
    "sobreprecio_pct_promedio"
].round(1)
resumen_clubes.rename(columns={"Club comprador_2": "Club"}, inplace=True)

# 4. Crear la visualización interactiva con Altair
grafico = (
    alt.Chart(resumen_clubes)
    .mark_bar()
    .encode(
        x=alt.X(
            "Sobreprecio (€M):Q",
            title="Sobreprecio Acumulado en Millones de Euros (€M)",
        ),
        y=alt.Y("Club:N", sort="-x", title="Club Comprador"),
        color=alt.Color(
            "Sobreprecio (€M):Q",
            scale=alt.Scale(scheme="reds"),
            legend=alt.Legend(title="Sobreprecio (€M)"),
        ),
        tooltip=[
            "Club",
            "Sobreprecio (€M)",
            "Total Pagado (€M)",
            "Valor Estimado Real (€M)",
            "Sobreprecio Promedio (%)",
            "total_fichajes",
        ],
    )
    .properties(
        title="El Impuesto Premier: Sobreprecio Acumulado Pagado por Club (2005-2026)",
        width=650,
        height=450,
    )
    .interactive()
)

# Guardar automáticamente los dos archivos que pide el encargo
grafico.save("vis_01.html")
grafico.save("vis_01.png")

# Mostrar el gráfico interactivo en pantalla
grafico


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 30.4 MB/s eta 0:00:00


alt.Chart(...)